# OptiCrop AI – Crop Recommendation Analysis
**SmartBridge Internship Project**

This notebook covers complete EDA, model training, evaluation, and comparison.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Libraries loaded!')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/Crop_recommendation.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print('Data Types:\n', df.dtypes)
print('\nMissing Values:\n', df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

In [ ]:
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
# Crop Distribution
plt.figure(figsize=(14,5))
df['label'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Crop Distribution in Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Crop'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Feature Distributions
features = ['N','P','K','temperature','humidity','ph','rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18,8))
axes = axes.flatten()
for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=30, color='teal', edgecolor='black', alpha=0.8)
    axes[i].set_title(feat, fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('Feature Histograms', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots
fig, axes = plt.subplots(2, 4, figsize=(18,8))
axes = axes.flatten()
for i, feat in enumerate(features):
    axes[i].boxplot(df[feat], patch_artist=True,
                    boxprops=dict(facecolor='lightgreen', color='darkgreen'))
    axes[i].set_title(feat, fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('Feature Boxplots (Outlier Detection)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10,7))
sns.heatmap(df[features].corr(), annot=True, cmap='YlGn', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Scatterplot: Temperature vs Humidity
plt.figure(figsize=(10,6))
crops = df['label'].unique()
for crop in crops[:10]:
    subset = df[df['label']==crop]
    plt.scatter(subset['temperature'], subset['humidity'], label=crop, alpha=0.6, s=20)
plt.xlabel('Temperature (°C)'); plt.ylabel('Humidity (%)')
plt.title('Temperature vs Humidity by Crop', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print('Classes:', list(le.classes_))

X = df[features].values
y = df['label_enc'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Model Training & Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes':         GaussianNB(),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }
    print(f'{name:22s} | Acc: {results[name]["accuracy"]:.4f} | F1: {results[name]["f1"]:.4f}')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Model Comparison Chart
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
f1s   = [results[n]['f1']       for n in names]
x = np.arange(len(names)); width = 0.35

fig, ax = plt.subplots(figsize=(12,5))
ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
ax.bar(x + width/2, f1s,  width, label='F1 Score',  color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0, 1.1); ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## 5. Best Model – Confusion Matrix & Feature Importance

In [ ]:
best_name  = max(results, key=lambda n: results[n]['accuracy'])
best_model = models[best_name]
print(f'Best Model: {best_name} ({results[best_name]["accuracy"]:.4f})')

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(16,12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix – {best_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance
rf = models['Random Forest']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,5))
plt.bar(range(len(features)), importances[indices], color='teal', edgecolor='black')
plt.xticks(range(len(features)), [features[i] for i in indices], rotation=15)
plt.title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

## 6. Save Model

In [ ]:
import os
os.makedirs('../model', exist_ok=True)
joblib.dump(best_model, '../model/model.pkl')
joblib.dump(scaler,     '../model/scaler.pkl')
joblib.dump(le,         '../model/label_encoder.pkl')
print('Model saved to ../model/')